# Phase 6 & 7: Test Set Inference, Validation & Submission Packaging
## Amazon ML Challenge 2026 — Multilingual Business Entity Resolution

This notebook executes:
1. **Partitioned Candidate Blocking** ($K \le 35$) across US, India, and France test sets.
2. **Vectorized Feature Extraction** (27 RapidFuzz & token features).
3. **LightGBM Match Classification** with calibrated decision threshold $\theta^* = 0.65$.
4. **TSV Output Generation** (`matching_results.tsv` and `candidate_pairs.tsv`).
5. **Submission Validation** (`validate_submission.py`).

In [ ]:
# Cell 1: Environment & Module Setup
import os
import sys
import gc
import time
import json
import subprocess
from pathlib import Path
from typing import Dict, List, Tuple
import polars as pl
import lightgbm as lgb

# Locate project root
current_path = Path.cwd().resolve()
while current_path.parent != current_path:
    if (current_path / "dataset").exists() or (current_path / "code").exists():
        project_root = current_path
        break
    current_path = current_path.parent
else:
    project_root = Path.cwd().resolve()

ber_code_dir = project_root / "code" / "business_entity_resolution"
if str(ber_code_dir) not in sys.path:
    sys.path.insert(0, str(ber_code_dir))

print(f"Project Root: {project_root}")
print(f"BER Code Dir: {ber_code_dir}")

from src.config import PROCESSED_DIR, OUTPUT_DIR, MODELS_DIR, SUBMISSION_MATCHING_TSV, SUBMISSION_CANDIDATE_TSV, VALIDATION_SCRIPT, TEST_DIR
from src.blocking import MultiIndexBlocker
from src.feature_engineering import FeatureExtractor

print("All modules imported successfully!")

In [ ]:
# Cell 2: Load Trained Model & Threshold Configuration
model_path = MODELS_DIR / "lgbm_entity_resolver.txt"
config_path = MODELS_DIR / "model_config.json"

print(f"Loading model from: {model_path}")
model = lgb.Booster(model_file=str(model_path))

with open(config_path, "r", encoding="utf-8") as f:
    model_config = json.load(f)

best_threshold = model_config.get("optimal_threshold", 0.65)
print(f"Loaded model config:")
print(f"  • Optimal Decision Threshold (θ*): {best_threshold:.2f}")
print(f"  • Validation Macro F0.5:           {model_config.get('best_macro_f05', 0.0):.4f}")
print(f"  • Validation Precision:            {model_config.get('best_precision', 0.0):.4f}")
print(f"  • Validation Recall:               {model_config.get('best_recall', 0.0):.4f}")

In [ ]:
# Cell 3: Load Test S1 Entities (Strict Ordering)
print("Reading cleaned test Source 1 dataset...")
t0 = time.time()
s1_test = pl.read_parquet(PROCESSED_DIR / "test_source1_cleaned.parquet")
all_s1_ids = s1_test["entity_id"].to_list()
total_s1 = len(all_s1_ids)

print(f"Loaded {total_s1:,} test S1 entities in {time.time() - t0:.2f}s")
print("Country distribution:")
for row in s1_test["country"].value_counts().iter_rows(named=True):
    print(f"  • {row['country']}: {row['count']:,}")

In [ ]:
# Cell 4: Execute Country-Partitioned Blocking & Inference
s1_to_candidates: Dict[str, List[str]] = {}
s1_to_matches: Dict[str, List[str]] = {}
countries = ["France", "India", "US"]
total_candidate_pairs_scored = 0
BATCH_SIZE = 100000

for country in countries:
    print("\n" + "=" * 70)
    print(f"PROCESSING COUNTRY PARTITION: {country.upper()}")
    print("=" * 70)
    t_country = time.time()

    # 1. Filter S1 for country
    s1_c = s1_test.filter(pl.col("country") == country)
    s1_c_ids = s1_c["entity_id"].to_list()
    print(f"  • S1 Entities: {len(s1_c_ids):,}")
    if len(s1_c_ids) == 0:
        continue

    # 2. Load country slice of S2 and S3 targets
    t_load = time.time()
    cols = ["entity_id", "country", "name_clean", "addr_clean", "name_tokens", "postal_digits", "has_address"]
    s2_c = pl.read_parquet(PROCESSED_DIR / "test_source2_cleaned.parquet", columns=cols).filter(pl.col("country") == country)
    s3_c = pl.read_parquet(PROCESSED_DIR / "test_source3_cleaned.parquet", columns=cols).filter(pl.col("country") == country)
    print(f"  • Loaded Targets in {time.time() - t_load:.2f}s: S2={len(s2_c):,}, S3={len(s3_c):,}")

    # 3. Fit MultiIndexBlocker
    blocker = MultiIndexBlocker(max_candidates=35, max_token_freq=4000)
    blocker.fit(s2_c, s3_c)

    # 4. Block S1
    t_block = time.time()
    cands_map = blocker.block_s1(s1_c)
    print(f"  • Candidate blocking done in {time.time() - t_block:.2f}s ({len(s1_c_ids)/(time.time() - t_block):,.0f} ent/s)")

    # 5. Extract Features & Predict
    extractor = FeatureExtractor()
    extractor.register_dataset(s1_c)
    extractor.register_dataset(s2_c)
    extractor.register_dataset(s3_c)

    pairs_to_score: List[Tuple[str, str]] = []
    for s1_id in s1_c_ids:
        cands = cands_map.get(s1_id, [])
        s1_to_candidates[s1_id] = cands
        for tgt_id in cands:
            pairs_to_score.append((s1_id, tgt_id))

    print(f"  • Pairs to score in {country}: {len(pairs_to_score):,}")
    total_candidate_pairs_scored += len(pairs_to_score)

    pair_probs: Dict[Tuple[str, str], float] = {}
    t_score = time.time()
    for idx in range(0, len(pairs_to_score), BATCH_SIZE):
        chunk = pairs_to_score[idx : idx + BATCH_SIZE]
        X_chunk, _ = extractor.extract_features_for_pairs(chunk)
        probs = model.predict(X_chunk)
        for pair, prob in zip(chunk, probs):
            pair_probs[pair] = float(prob)
        if (idx // BATCH_SIZE) % 5 == 0 or idx + BATCH_SIZE >= len(pairs_to_score):
            print(f"    Scored {min(idx + BATCH_SIZE, len(pairs_to_score)):,} / {len(pairs_to_score):,} pairs...")

    print(f"  • Scoring finished in {time.time() - t_score:.2f}s ({len(pairs_to_score)/(time.time() - t_score):,.0f} pairs/s)")

    # 6. Apply threshold
    for s1_id in s1_c_ids:
        cands = s1_to_candidates.get(s1_id, [])
        matched = [tgt for tgt in cands if pair_probs.get((s1_id, tgt), 0.0) >= best_threshold]
        s1_to_matches[s1_id] = matched

    # Clean RAM
    del s2_c, s3_c, blocker, extractor, pairs_to_score, pair_probs, cands_map
    gc.collect()
    print(f"  Country {country} partition finished in {time.time() - t_country:.2f}s")

In [ ]:
# Cell 5: Write Output TSVs in Exact S1 Order
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
matching_out = SUBMISSION_MATCHING_TSV
candidate_out = SUBMISSION_CANDIDATE_TSV

print(f"Writing official submission TSVs:")
print(f"  • {matching_out}")
print(f"  • {candidate_out}")

singletons_count = 0
total_predicted_matches = 0

with open(matching_out, "w", encoding="utf-8") as f_match, open(candidate_out, "w", encoding="utf-8") as f_cand:
    # Exact headers required by official validator
    f_match.write("source1_entity_id\tmatched_entity_ids\n")
    f_cand.write("source1_entity_id\tcandidate_entity_ids\n")

    for s1_id in all_s1_ids:
        cands = s1_to_candidates.get(s1_id, [])
        matches = s1_to_matches.get(s1_id, [])
        
        f_cand.write(f"{s1_id}\t{','.join(cands)}\n")
        if matches:
            f_match.write(f"{s1_id}\t{','.join(matches)}\n")
            total_predicted_matches += len(matches)
        else:
            f_match.write(f"{s1_id}\t\n")
            singletons_count += 1

print("\n--- INFERENCE SUMMARY ---")
print(f"Total Test S1 Entities:   {total_s1:,}")
print(f"Total Candidate Pairs:    {total_candidate_pairs_scored:,} (Avg {total_candidate_pairs_scored/total_s1:.1f}/ent)")
print(f"Total Matches Predicted:  {total_predicted_matches:,} (Avg {total_predicted_matches/total_s1:.2f}/ent)")
print(f"Singletons Defended:      {singletons_count:,} ({singletons_count/total_s1*100:.2f}%)")

In [ ]:
# Cell 6: Run Official Submission Validator
print("Running official submission validation harness...")
val_cmd = [
    sys.executable,
    str(VALIDATION_SCRIPT),
    "--matching", str(matching_out),
    "--candidate", str(candidate_out),
    "--test-dir", str(TEST_DIR),
    "--check-ids",
]

res = subprocess.run(val_cmd, capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print(res.stderr)

if res.returncode == 0:
    print("\n[SUCCESS] 100% compliant with competition rules!")
else:
    print(f"\n[FAILED] Validator returned code: {res.returncode}")